In [13]:
%pip install pypsa 
%pip install highs
%pip install gurobi

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: pypsa in c:\users\user\appdata\roaming\python\python312\site-packages (1.2.2)



Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement highs (from versions: none)
ERROR: No matching distribution found for highs


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement gurobi (from versions: none)
ERROR: No matching distribution found for gurobi


In [14]:
import pypsa 
import pandas as pd

# Отключаем использование экспериментальных Arrow-типов для строк
pd.options.future.infer_string = False

In [15]:
network = pypsa.Network(snapshots=pd.Index([0]))

# Пример 1: Экономическая диспетчеризация на одной шине

В этом примере соберем простую сеть с одним узлом (bus), несколькими генераторами и нагрузкой. Решим задачу минимизации затрат на производство электроэнергии, что бы покрыть спрос ∑ 
g
​
 c 
g
​
 ⋅p 
g

Минимизацией не занимается PyPSA, а специальный решатель (solver), обычно HiGHS или GLPK

In [16]:
network.add("Bus", 'Main Bus')

network.add("Generator", 
    name = ['Solar', 'Wind', 'Gas'],
    bus = "Main Bus",
    p_nom = [100, 150, 300], # максимальная мощность (МВт)
    marginal_cost = [0, 0.01, 60]
)

network.add("Load", name = 'Demand', bus = 'Main Bus', p_set = 200)

In [17]:
# запускаем оптимизацию
network.optimize()

print("Power dispatched from each generator [MW]:")
print(network.generators_t.p)

C:\Users\User\AppData\Local\Temp\ipykernel_28572\2952962744.py:2: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  network.optimize()
Index(['Main Bus'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.03s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 3 primals, 7 duals
Objective: 1.00e+00
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper were not assigned to the network.


Power dispatched from each generator [MW]:
name      Solar   Wind  Gas
snapshot                   
0         100.0  100.0 -0.0


# Пример 2: Линейный оптимальный поток мощности в сети с несколькими узлами.

Теперь добавим второй узел и соеденим их линией с ограничением пропускной способности (s_nom). Здесь в игру вступают физические законы. Pypsa будет считать не только баланс генерации и спроса, но и решать физику: по каким линиями и сколько потеет мощноти, чтот бы минимизировать стоимость, но без перегрузки.

In [27]:
network_lopf = pypsa.Network(snapshots = pd.Index([0]))

In [29]:
network_lopf.add("Bus", name = 'Bus A')
network_lopf.add('Generator', name = ['wind', 'solar', 'coal', 'gas' ], bus = 'Bus A', p_nom = [40, 50, 100, 100], marginal_cost = [0,0,20,60])

network_lopf.add("Load", name = "Load", bus = "Bus A", p_set = [160])

In [31]:
print(network_lopf.consistency_check())

Index(['Bus A'], dtype='object', name='name')


None


In [32]:
network_lopf.optimize()

C:\Users\User\AppData\Local\Temp\ipykernel_28572\573910852.py:1: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  network_lopf.optimize()
Index(['Bus A'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.03s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 4 primals, 9 duals
Objective: 1.40e+03
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper were not assigned to the network.


('ok', 'optimal')

In [36]:
print("\nГенерация (МВт):")
print(network_lopf.generators_t.p)
print(network_lopf.buses_t.marginal_price)


Генерация (МВт):
name      wind  solar  coal  gas
snapshot                        
0         40.0   50.0  70.0 -0.0
name      Bus A
snapshot       
0          20.0
